# Tutorial 06 -- Qubit Spectroscopy

Perform a weak-drive qubit spectroscopy sweep in the matched rotating frame and fit the resonance with a Lorentzian model.

**Prerequisites.** Tutorials 01, 02, and 04 are recommended first.


## 1. Goal

We will sweep a weak qubit probe around the rotating-frame resonance, read out the final excited-state population, and fit the peak position.


## 2. Physical Background

A long, weak drive is the simplest spectroscopy experiment. In the matched rotating frame the bare qubit line appears near zero detuning, and the probe frequency is specified as a physical transition detuning before being converted to an internal waveform carrier.


## 3. Imports


In [1]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    ns,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [2]:
detuning_mhz = np.linspace(-3.0, 3.0, 61)
probe_duration = 1.0 * us
probe_amplitude = MHz(0.08)
dt = 4.0 * ns


## 5. Model Construction


In [3]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.0),
    omega_q=GHz(6.2),
    alpha=0.0,
    chi=0.0,
    kerr=0.0,
    n_cav=1,
    n_tr=2,
)
frame = FrameSpec(omega_q_frame=model.omega_q)


## 6. Pulse / Sequence Construction


In [4]:
responses = []
for point_mhz in detuning_mhz:
    probe = Pulse(
        "q",
        t0=0.0,
        duration=probe_duration,
        envelope=square_envelope,
        carrier=carrier_for_transition_frequency(MHz(point_mhz)),
        amp=probe_amplitude,
        label="spectroscopy_probe",
    )
    compiled = SequenceCompiler(dt=dt).compile([probe], t_end=probe_duration + dt)
    result = simulate_sequence(
        model,
        compiled,
        model.basis_state(0, 0),
        {"q": "qubit"},
        config=SimulationConfig(frame=frame, max_step=dt),
    )
    responses.append(final_expectation(result, "P_e"))
responses = np.asarray(responses, dtype=float)


ValueError: number of diagonals does not match number of offsets

## 7. Running the Simulation


In [ ]:
fit = fit_lorentzian_peak(detuning_mhz, responses)
print("Fitted spectroscopy center [MHz]:", fit.parameters["center"])
print("Fitted width [MHz]:", fit.parameters["width"])


## 8. Visualizing the Results


In [ ]:
fig, ax = plt.subplots()
ax.plot(detuning_mhz, responses, "o", label="simulation")
ax.plot(detuning_mhz, fit.model_y, "-", label="Lorentzian fit")
ax.axvline(fit.parameters["center"], color="tab:red", linestyle="--", label="fit center")
ax.set_xlabel("Transition detuning relative to the qubit frame [MHz]")
ax.set_ylabel(r"Final $P_e$")
ax.set_title("Weak-drive qubit spectroscopy")
ax.legend()
plt.show()


## 9. Physical Interpretation

The resonance appears near zero because the frame matches the bare qubit frequency. The key convention detail is that the x-axis is a physical transition detuning, while the simulator receives the internal raw carrier through `carrier_for_transition_frequency(...)`.


## 10. Exercises / Next Steps

- Repeat the scan with a deliberate frame offset and see how the fitted center moves.
- Increase the probe amplitude until power broadening becomes obvious.
- Continue to Tutorial 07 for photon-number-resolved spectroscopy.
